In [ ]:
# connect drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# import libraries
from pathlib import Path
import json

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

In [ ]:
%cd /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/notebooks

/content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/notebooks


In [ ]:
Path.cwd()

PosixPath('/content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/notebooks')

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("project root:", PROJECT_ROOT)
print("data dir:", DATA_DIR)
print("processed dir:", PROCESSED_DIR)

project root: /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation
data dir: /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/data
processed dir: /content/drive/MyDrive/english-french-machine-translation/english-french-machine-translation/data/processed


In [ ]:
# load saved tensors
data = torch.load(
    PROCESSED_DIR / "translation_ids.pt",
    map_location=torch.device("cpu"),
    weights_only=True,
)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]


In [ ]:

from tokenizers import Tokenizer

english_tokenizer = Tokenizer.from_file(
    str(PROCESSED_DIR / "english_tokenizer.json")
)

french_tokenizer = Tokenizer.from_file(
    str(PROCESSED_DIR / "french_tokenizer.json")
)

with open(PROCESSED_DIR / "preprocessing_config.json", "r") as file:
    config = json.load(file)


In [ ]:

EN_VOCAB_SIZE = english_tokenizer.get_vocab_size()
FR_VOCAB_SIZE = french_tokenizer.get_vocab_size()

PAD_ID = french_tokenizer.token_to_id("[PAD]")
BOS_ID = french_tokenizer.token_to_id("[BOS]")
EOS_ID = french_tokenizer.token_to_id("[EOS]")

print("Training tensors:", X_train.shape, y_train.shape)
print("Validation tensors:", X_val.shape, y_val.shape)
print("Test tensors:", X_test.shape, y_test.shape)

print("\nEnglish vocabulary size:", EN_VOCAB_SIZE)
print("French vocabulary size:", FR_VOCAB_SIZE)

print("\nFrench special-token IDs:")
print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

print("\nPreprocessing configuration:")
print(config)


Training tensors: torch.Size([133704, 20]) torch.Size([133704, 22])
Validation tensors: torch.Size([16713, 20]) torch.Size([16713, 22])
Test tensors: torch.Size([16713, 20]) torch.Size([16713, 22])

English vocabulary size: 14493
French vocabulary size: 23539

French special-token IDs:
PAD: 1
BOS: 2
EOS: 3

Preprocessing configuration:
{'english_max_length': 20, 'french_max_length': 22, 'english_vocab_size': 14493, 'french_vocab_size': 23539, 'pad_token': '[PAD]', 'unk_token': '[UNK]', 'bos_token': '[BOS]', 'eos_token': '[EOS]'}


In [ ]:
EN_VOCAB_SIZE = english_tokenizer.get_vocab_size()
EN_PAD_ID = english_tokenizer.token_to_id("[PAD]")

FR_VOCAB_SIZE = french_tokenizer.get_vocab_size()
FR_PAD_ID = french_tokenizer.token_to_id("[PAD]")

print("English vocabulary:", EN_VOCAB_SIZE)
print("English padding ID:", EN_PAD_ID)
print("French vocabulary:", FR_VOCAB_SIZE)
print("French padding ID:", FR_PAD_ID)

English vocabulary: 14493
English padding ID: 1
French vocabulary: 23539
French padding ID: 1


In [ ]:
# encoder class

class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        padding_idx
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
            )

        packed_outputs, hidden = self.gru(packed)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs, batch_first=True,
            total_length=input_ids.shape[1]
        )

        return outputs, hidden


In [ ]:
train_en_lengths = (X_train != EN_PAD_ID).sum(dim=1)
val_en_lengths = (X_val != EN_PAD_ID).sum(dim=1)
test_en_lengths = (X_test != EN_PAD_ID).sum(dim=1)

print("First 5 English lengths:", train_en_lengths[:5])
print("First 5 English sequences:", X_train[:5].shape)

First 5 English lengths: tensor([ 8, 16, 11, 11,  9])
First 5 English sequences: torch.Size([5, 20])


In [ ]:
encoder = Encoder(
    vocab_size=EN_VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=1,
    padding_idx=EN_PAD_ID,
)

outputs, hidden = encoder(
    X_train[:4],
    train_en_lengths[:4],
)

print("Encoder outputs:", outputs.shape)
print("Encoder hidden state:", hidden.shape)

Encoder outputs: torch.Size([4, 20, 256])
Encoder hidden state: torch.Size([1, 4, 256])


In [ ]:
# Decoder class

class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        padding_idx,
        dropout_rate=0.2
    ):
        super().__init__()

        # convert French token IDs into vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )

        # process the French sequence
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,

        )

        # Dropout before the output projection
        self.dropout = nn.Dropout(dropout_rate)

        # convert each GRU output into vocabulary scores
        self.fc = nn.Linear(
            in_features=hidden_dim,
            out_features=vocab_size,
        )

    def forward(self, input_ids, hidden):
        # (batch_size, target_length)
        embedded = self.embedding(input_ids)

        # (batch_size, target_length, embedding_dim)
        outputs, hidden = self.gru(embedded, hidden)

        # Apply dropout to GRU outputs
        outputs = self.dropout(outputs)

        # (batch_size, target_length, vocab_size)
        logits = self.fc(outputs)

        return logits, hidden



In [ ]:
# Seq2Seq class


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source_ids, source_lengths, target_ids):
        # Encode the English sentences
        encoder_outputs, encoder_hidden = self.encoder(
            source_ids,
            source_lengths,
        )

        # Use the encoder's final hidden state to initialize the decoder
        logits, decoder_hidden = self.decoder(
            target_ids,
            encoder_hidden,
        )

        return logits


In [ ]:

encoder = Encoder(
    vocab_size=EN_VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=1,
    padding_idx=EN_PAD_ID,
)

decoder = Decoder(
    vocab_size=FR_VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=1,
    padding_idx=FR_PAD_ID,
    dropout_rate=0.2,
)

model = Seq2Seq(
    encoder=encoder,
    decoder=decoder,
)


In [ ]:

source_ids = X_train[:4]
source_lengths = train_en_lengths[:4]

decoder_input = y_train[:4, :-1]

logits = model(
    source_ids,
    source_lengths,
    decoder_input,
)

print("Decoder input:", decoder_input.shape)
print("Model logits:", logits.shape)


Decoder input: torch.Size([4, 21])
Model logits: torch.Size([4, 21, 23539])


In [ ]:
# switch to device to cuda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [22]:
%%writefile -a utils.py
# training and evaluation function
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    num_epochs,
    EN_PAD_ID,
    FR_PAD_ID,
    device
):
    model.to(device)

    train_losses = [],
    val_losses = [],

    train_accuracies = [],
    val_accuracies = [],
    learning_rates = [],

    #--------------------------------
    # Training
    #--------------------------------

    for epoch in range(num_epochs):
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for source_ids, target_ids in train_loader:


            source_ids = source_ids.to(device)
            target_ids = target_ids.to(device)


            decoder_input = target_ids[:, :-1]
            expected = target_ids[:, 1:]
            source_lengths = (source_ids != EN_PAD_ID).sum(dim=1)

            optimizer.zero_grad()

            logits = model(
                source_ids,
                source_lengths,
                decoder_input,
            )

            loss = criterion(logits.transpose(1, 2), expected)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            valid = expected != FR_PAD_ID
            train_loss += loss.item() * valid.sum().item()
            train_correct += (
                (logits.argmax(dim=-1) == expected) & valid
                ).sum().item()

            train_total = valid.sum().item()


        train_losses.append(train_loss / train_total)
        train_accuracies.append(train_correct / train_total)

        # ---------------------------------------------------
        # Validation
        #----------------------------------------------------
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for source_ids, target_ids in val_loader:
                source_ids = source_ids.to(device)
                target_ids = target_ids.to(device)

                source_lengths = (source_ids != EN_PAD_ID).sum(dim=1)

                decoder_input = target_ids[:, :-1]
                expected = target_ids[:, 1:]

                logits = model(
                    source_ids,
                    source_lengths,
                    decoder_input,
                )

                loss = criterion(logits.transpose(1, 2), expected)

                valid = expected != FR_PAD_ID
                val_loss += loss.item() * valid.sum().item()
                val_correct += (
                    (logits.argmax(dim=-1) == expected) & valid
                    ).sum().item()
                val_total = valid.sum().item()

        val_losses.append(val_loss / val_total)
        val_accuracies.append(val_correct / val_total)

        # ----------------------------------------------
        # Learning rate schedule
        # ----------------------------------------------

        scheduler.step(val_losses[-1])
        learning_rates.append(optimizer.param_groups[0]["lr"])

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"Train Loss: {train_losses[-1]:.4f} | "
            f"Val Loss: {val_losses[-1]:.4f} | "
            f"Train Acc: {train_accuracies[-1]:.4f} | "
            f"Val Acc: {val_accuracies[-1]:.4f} | "
            f"LR: {learning_rates[-1]:.6f}"
        )

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accuracies": train_accuracies,
        "val_accuracies": val_accuracies,
        "learning_rates": learning_rates,
    }





Appending to utils.py


In [21]:
%%writefile -a utils.py
# plotting function
import matplotlib.pyplot as plt

def plot_training_history(history):
    epochs = range(1, len(history["train_losses"]) + 1)

     # 1. Loss
    plt.figure(figsize=(12, 5))
    plt.plot(epochs, history["train_losses"], label="Training Loss")
    plt.plot(epochs, history["val_losses"], label="Validation Loss")
    plt.title("Training and Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    # 2. Acurracy
    plt.figure(figsize=(12, 5))
    plt.plot(epochs, history["train_accuracies"], label="Training Accuracy")
    plt.plot(epochs, history["val_accuracies"], label="Validation Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # 3. Learning rate
    plt.figure(figsize=(12, 5))
    plt.plot(epochs, history["learning_rates"], label="Learning Rate")
    plt.title("Learning Rate Schedule")
    plt.xlabel("Epochs")
    plt.ylabel("Learning Rate")
    plt.legend()
    plt.grid(True)
    plt.yscale("log")
    plt.show()

#

Appending to utils.py


In [23]:
%%writefile -a utils.py
def evaluate_model(
    model,
    test_loader,
    criterion,
    device,
    en_pad_id,
    fr_pad_id,
):
    """
    Evaluate a trained Seq2Seq model on the test dataset.

    Returns:
        Dictionary containing test loss and token accuracy.
    """

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():

        for source_ids, target_ids in test_loader:

            source_ids = source_ids.to(device)
            target_ids = target_ids.to(device)

            # Calculate the actual English sequence lengths.
            source_lengths = (source_ids != en_pad_id).sum(dim=1)

            # Teacher forcing: shift the French target by one position.
            decoder_input = target_ids[:, :-1]
            expected = target_ids[:, 1:]

            # Forward pass.
            logits = model(
                source_ids,
                source_lengths,
                decoder_input,
            )

            # Count only non-padding target tokens.
            valid_tokens = expected != fr_pad_id
            num_tokens = valid_tokens.sum().item()

            # CrossEntropyLoss expects (batch, classes, sequence).
            loss = criterion(
                logits.transpose(1, 2),
                expected,
            )

            total_loss += loss.item() * num_tokens

            # Get the predicted token at each position.
            predictions = logits.argmax(dim=-1)

            # Ignore padding when calculating accuracy.
            correct = (
                (predictions == expected) & valid_tokens
            ).sum().item()

            total_correct += correct
            total_tokens += num_tokens

    test_loss = total_loss / total_tokens
    test_accuracy = total_correct / total_tokens

    results = {
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
    }

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Token Accuracy: {test_accuracy:.2%}")

    return results


Appending to utils.py


In [24]:
# create tensor dataset and dataloader

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [25]:
print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

2090
262
262


In [26]:
!git status

Refresh index: 100% (13/13), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   01_text_preprocessing.ipynb
	modified:   02_gru_translation.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	utils.py

no changes added to commit (use "git add" and/or "git commit -a")


In [27]:
!git add notebooks/02_gru_translation.ipynb src/utils.py
!git commit -m "Add GRU data loaders and training utilities"
!git push origin main

fatal: pathspec 'notebooks/02_gru_translation.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   01_text_preprocessing.ipynb
	modified:   02_gru_translation.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	utils.py

no changes added to commit (use "git add" and/or "git commit -a")
fatal: could not read Username for 'https://github.com': No such device or address
